# Trial Eligibility

**Assess number of patients in the aNSCLC analytic cohort meeting PERSEE ECOG and lab eligibility.**

In [1]:
import sys
sys.path.append('../..')

import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorNSCLC

## Import dfs

In [2]:
index_df = pd.read_csv('../outputs/pembrochemo_pembro_index.csv')

In [3]:
index_df.shape

(20572, 3)

In [4]:
dtype_map = pd.read_csv('../outputs/pembrochemo_pembro_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/pembrochemo_pembro_features_df.csv', dtype = dtype_map)

In [5]:
features_df.shape

(6601, 169)

In [6]:
surv_pred_df = pd.read_csv('../outputs/gb_6m_survival_predictions_calibrated.csv')

In [7]:
surv_pred_df.shape

(5605, 2)

In [8]:
main_df = pd.merge(features_df, index_df, on = 'PatientID', how = 'left')

In [9]:
main_df = pd.merge(main_df, surv_pred_df, on = 'PatientID', how = 'left')

In [10]:
main_df.shape

(6601, 172)

In [11]:
main_df['StartDate'] = pd.to_datetime(main_df['StartDate'])

In [12]:
main_df['treatment_year'] = main_df['StartDate'].dt.year

In [13]:
main_df = main_df.query('treatment_year <= 2023')

In [14]:
main_df.shape

(5605, 173)

In [15]:
main_df = main_df[['PatientID', 'LineName', 'StartDate', 'psurv_180_calibrated']]

In [16]:
main_df.head(3)

,PatientID,LineName,StartDate,psurv_180_calibrated
0,FF16F972863F8,pembro_platinum,2019-01-18,0.399760
1,F5EF114860555,pembro_platinum,2022-09-30,0.543518
3,F95B93B796545,pembro_platinum,2019-10-08,0.926144


## ECOG and Labs

In [17]:
# Initialize class 
processor = DataProcessorNSCLC()

In [18]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = main_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-11-26 15:11:29,595 - INFO - Successfully read Lab.csv file with shape: (82093498, 17) and unique PatientIDs: 108761
2026-11-26 15:11:48,560 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (5481771, 18) and unique PatientIDs: 5543
2026-11-26 15:12:01,007 - INFO - Successfully processed Lab.csv file with final shape: (5605, 81) and unique PatientIDs: 5605


In [19]:
labs_df = labs_df[['PatientID', 'anc', 'hemoglobin', 'platelet', 'creatinine', 'total_bilirubin', 'ast', 'alt', 'alp']]

In [20]:
labs_df.shape

(5605, 9)

In [21]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = main_df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-11-26 15:12:01,652 - INFO - Successfully read ECOG.csv file with shape: (1623197, 4) and unique PatientIDs: 85577
2026-11-26 15:12:01,886 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (122080, 5) and unique PatientIDs: 4811
2026-11-26 15:12:01,978 - INFO - Successfully processed ECOG.csv file with final shape: (5605, 3) and unique PatientIDs: 5605


In [22]:
ecog_df = ecog_df[['PatientID', 'ecog_index']]

In [23]:
ecog_df.shape

(5605, 2)

In [24]:
main_df = pd.merge(main_df, labs_df, on = 'PatientID', how = 'left')

In [25]:
main_df = pd.merge(main_df, ecog_df, on = 'PatientID', how = 'left')

In [26]:
main_df.shape

(5605, 13)

In [27]:
main_df.head(3)

,PatientID,LineName,StartDate,psurv_180_calibrated,anc,hemoglobin,platelet,creatinine,total_bilirubin,ast,alt,alp,ecog_index
0,FF16F972863F8,pembro_platinum,2019-01-18,0.399760,7.1,11.2,191.0,1.1,0.4,15.0,10.0,54.0,0
1,F5EF114860555,pembro_platinum,2022-09-30,0.543518,NaN,11.8,286.0,1.1,0.3,15.0,17.0,75.0,NaN
2,F95B93B796545,pembro_platinum,2019-10-08,0.926144,6.5,12.6,363.0,0.7,0.5,12.0,12.0,102.0,0


## Risk stratification

In [28]:
with open('../outputs/crossover_survival_estimate.txt', 'r') as f:
    crossover_survival_estimate = float(f.read())

In [29]:
main_df['high_risk'] = np.where(main_df['psurv_180_calibrated'] < crossover_survival_estimate, 1, 0)

In [30]:
main_df.high_risk.value_counts()

high_risk
0    4008
1    1597
Name: count, dtype: int64

## ECOG Eligibility

In [31]:
main_df['ecog_eligible'] = np.where((main_df['ecog_index'] <= 2), 1, 0)

In [32]:
main_df['ecog_ineligible'] = np.where(main_df['ecog_index'] > 2, 1, 0)

In [33]:
main_df['ecog_indeterminate'] = np.where(main_df['ecog_index'].isna(), 1, 0)

In [34]:
main_df.shape[0]

5605

In [35]:
main_df.ecog_eligible.value_counts()[1]+main_df.ecog_ineligible.value_counts()[1]+main_df.ecog_indeterminate.value_counts()[1]

np.int64(5605)

## Lab eligibility

In [36]:
main_df['lab_eligible'] = np.where(
    (main_df['anc'] >= 1.5) &
    (main_df['platelet'] >= 100) &
    (main_df['hemoglobin'] >= 8) &
    (main_df['creatinine'] <= 1.8) & 
    (main_df['total_bilirubin'] <= 1.8) &
    (main_df['ast'] <= 100) &
    (main_df['alt'] <= 100) &
    (main_df['alp'] <= 300), 1, 0)

In [37]:
main_df['lab_ineligible'] = np.where(
    (main_df['anc'] < 1.5) |
    (main_df['platelet'] < 100) |
    (main_df['hemoglobin'] < 8) |
    (main_df['creatinine'] > 1.8) | 
    (main_df['total_bilirubin'] > 1.8) |
    (main_df['ast'] > 100) |
    (main_df['alt'] > 100) |
    (main_df['alp'] > 300), 1, 0)

In [38]:
main_df['lab_indeterminate'] = np.where((main_df['lab_eligible'] == 0) & (main_df['lab_ineligible'] == 0), 1, 0)

In [39]:
main_df.lab_eligible.value_counts()[1]+main_df.lab_ineligible.value_counts()[1]+main_df.lab_indeterminate.value_counts()[1]

np.int64(5605)

## Overal eligibility

In [40]:
main_df['eligible'] = np.where((main_df['ecog_eligible'] == 1) & (main_df['lab_eligible'] == 1), 1, 0)

In [41]:
main_df['ineligible'] = np.where((main_df['ecog_ineligible'] == 1) | (main_df['lab_ineligible'] == 1), 1, 0)

In [42]:
main_df['indeterminate'] = np.where((main_df['eligible'] == 0) & (main_df['ineligible'] == 0), 1, 0)

In [43]:
print('Overall cohort')
print(f'Eligible: count {main_df.eligible.value_counts()[1]}, percentage {round(main_df.eligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Ineligible: count {main_df.ineligible.value_counts()[1]}, percentage {round(main_df.ineligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Indeterminate: count {main_df.indeterminate.value_counts()[1]}, percentage {round(main_df.indeterminate.value_counts(normalize = True)[1]*100, 2)}')

print('')
print('High-risk')
print(f'Eligible: count {main_df.query('high_risk == 1').eligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 1').eligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Ineligible: count {main_df.query('high_risk == 1').ineligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 1').ineligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Indeterminate: count {main_df.query('high_risk == 1').indeterminate.value_counts()[1]}, percentage {round(main_df.query('high_risk == 1').indeterminate.value_counts(normalize = True)[1]*100, 2)}')

print('')
print('Low-risk')
print(f'Eligible: count {main_df.query('high_risk == 0').eligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 0').eligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Ineligible: count {main_df.query('high_risk == 0').ineligible.value_counts()[1]}, percentage {round(main_df.query('high_risk == 0').ineligible.value_counts(normalize = True)[1]*100, 2)}')
print(f'Indeterminate: count {main_df.query('high_risk == 0').indeterminate.value_counts()[1]}, percentage {round(main_df.query('high_risk == 0').indeterminate.value_counts(normalize = True)[1]*100, 2)}')

Overall cohort
Eligible: count 2826, percentage 50.42
Ineligible: count 623, percentage 11.12
Indeterminate: count 2156, percentage 38.47

High-risk
Eligible: count 704, percentage 44.08
Ineligible: count 419, percentage 26.24
Indeterminate: count 474, percentage 29.68

Low-risk
Eligible: count 2122, percentage 52.94
Ineligible: count 204, percentage 5.09
Indeterminate: count 1682, percentage 41.97
